# Experiment 4: Hill-Climbing Search for the Eight Queens Problem
**Course:** CSAI4014 Algorithms for Intelligent Systems and Robotics  
**Student:** Aadi Chopra | SAP ID: 500120163 | Roll No: R2142231902 | Batch: B2 AIML  
**Repository:** https://github.com/aadichopra/aisr-lab-500120163  

## 1. Aim
To implement steepest-ascent hill climbing on the 8-queens problem, analyze the objective landscape and local optima, and evaluate the success rate of random-restart hill climbing as a function of the number of restarts $k$.


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

os.makedirs('figs', exist_ok=True)

def compute_conflicts(state):
    conflicts = 0; n = len(state)
    for i in range(n):
        for j in range(i + 1, n):
            if state[i] == state[j] or abs(state[i] - state[j]) == abs(i - j):
                conflicts += 1
    return conflicts

def hill_climb_single(max_steps=100, rng=None):
    if rng is None: rng = np.random.default_rng()
    current = list(rng.integers(0, 8, size=8))
    current_h = compute_conflicts(current)
    steps = 0; landscape = [current_h]

    for step in range(max_steps):
        if current_h == 0: break
        best_state = None; best_h = current_h
        for col in range(8):
            orig_row = current[col]
            for row in range(8):
                if row == orig_row: continue
                nbr = list(current); nbr[col] = row
                h = compute_conflicts(nbr)
                if h < best_h:
                    best_h = h; best_state = nbr
        if best_state is None: break
        current, current_h = best_state, best_h
        steps += 1; landscape.append(current_h)
    return current, current_h, steps, landscape

# Run 500 trials of plain hill climbing
rng = np.random.default_rng(42)
num_trials = 500
successes = 0
steps_list = []
trajectories = []

for i in range(num_trials):
    state, h, steps, traj = hill_climb_single(rng=rng)
    if h == 0: successes += 1
    steps_list.append(steps)
    if i < 20: trajectories.append((traj, h == 0))

p_base = successes / num_trials
print(f"Trials: {num_trials}, Successes: {successes}, Base Success Rate p = {p_base*100:.2f}%")
print(f"Mean Steps: {np.mean(steps_list):.2f}, Max Steps: {max(steps_list)}")

# Sweep random restarts k = 1 to 20
k_values = [1, 2, 3, 5, 8, 10, 15, 20]
empirical_success = []
theoretical_success = [1.0 - (1.0 - p_base)**k for k in k_values]

for k in k_values:
    restarts_success = 0
    test_runs = 200
    for _ in range(test_runs):
        for _ in range(k):
            _, h, _, _ = hill_climb_single(rng=rng)
            if h == 0:
                restarts_success += 1
                break
    empirical_success.append(restarts_success / test_runs)

# Plot Figure 4.1: Objective Landscape Trajectories
fig, ax = plt.subplots(figsize=(6.5, 3.6), dpi=150)
for traj, succ in trajectories:
    color = '#2ca02c' if succ else '#d62728'
    alpha = 0.8 if succ else 0.4
    ax.plot(range(len(traj)), traj, marker='o', markersize=3, color=color, alpha=alpha)

ax.set_xlabel('Search Step')
ax.set_ylabel('Objective Value (Attacking Queen Pairs $h$)')
ax.set_title('Hill-Climbing Trajectories: Convergence to Global (h=0) vs. Local Optima (h>0)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.6)
ax.plot([], [], color='#2ca02c', label='Success (Global Optimum $h=0$)')
ax.plot([], [], color='#d62728', label='Stuck at Local Optimum ($h > 0$)')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('figs/fig4_1_objective_landscape.png')
plt.show()

# Plot Figure 4.2: Success Rate vs Restarts k
fig, ax = plt.subplots(figsize=(6.2, 3.6), dpi=150)
ax.plot(k_values, [s * 100 for s in empirical_success], 'bo-', label='Empirical Multi-Start Success Rate (%)')
ax.plot(k_values, [s * 100 for s in theoretical_success], 'r--', label='Theoretical Model: $1 - (1-p)^k$')
ax.set_xlabel('Number of Random Restarts $k$')
ax.set_ylabel('Success Rate (%)')
ax.set_title('Random-Restart Hill Climbing Success Rate (p = 13.2%)', fontsize=11)
ax.grid(True, linestyle='--', alpha=0.6)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('figs/fig4_2_restarts_success.png')
plt.show()
